# B8 · Buffers and nearest features: how close is close?

> **Time:** about 90 minutes  
> **You need:** Part A and Lessons B1 to B7. You should be able to convert layers with `to_crs(32645)`, make a line from the trek stops (B2), and select with spatial predicates (B7).  
> **By the end you can:**
> 1. draw **buffers** of a set distance around points and lines, in metres, and explain why they must not be drawn in degrees or Web Mercator,
> 2. control the shape of a buffer (`quad_segs`, `cap_style`, single-sided buffers) and merge overlapping buffers with `union_all()`,
> 3. find which landslides lie within 10 km of a district headquarters, and the nearest headquarters to every landslide with `gpd.sjoin_nearest`.

Many real questions are about distance. Which villages are within 10 km of a health post? Which landslides are close to a district headquarters, where help would come from? Which is the nearest town to each landslide, and how far away is it? This lesson answers such questions with **buffers** and **nearest-feature joins**, always measured in metres.

**Counting lines in this lesson:** the lines that only set up a plot, `fig, ax = plt.subplots()` and `plt.show()`, are not counted in the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 18 of 21 · [Course home](../README.md)  
← Previous: [B7 · Clip and select by location: inside, touching, cut to shape](B07_Clip_and_select_by_location.ipynb) · Next: [B9 · Spatial joins and overlays: landslides, districts and headquarters](B09_Spatial_joins_and_overlays.ipynb) →

## 1. A buffer: everything within a distance

A **buffer** is the area within a given distance of a feature. Around a point it is a circle, around a line a corridor with rounded ends, and around a polygon a larger polygon.

`gdf.buffer(distance)` draws one buffer per feature. The distance is in the **units of the CRS**. So first convert to UTM zone 45N (EPSG:32645), whose unit is the metre, as in B4. Then `buffer(10000)` means 10 km.

`.buffer()` returns a **GeoSeries** of polygons, one per feature, without the other columns.

**Predict:** a circle of radius 10 km has an area of π × 10² ≈ 314.2 km². What will the buffer's area be?

In [ ]:
import math
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import LineString
from pathlib import Path

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

districts = gpd.read_file(DATA / "nepal_districts.geojson")
hq = gpd.read_file(DATA / "district_headquarters.geojson")
slides_table = pd.read_csv(DATA / "practice_landslides.csv")
slides = gpd.GeoDataFrame(slides_table,
                          geometry=gpd.points_from_xy(slides_table["lon"], slides_table["lat"]),
                          crs="EPSG:4326")
slides["district"] = slides["district"].replace({"Chitwan": "Chitawan"})   # the fix from B6

# everything that will be measured goes to UTM 45N, in metres
districts_utm = districts.to_crs(32645)
hq_utm = hq.to_crs(32645)
slides_utm = slides.to_crs(32645)
print(hq_utm.crs.axis_info[0].unit_name)

In [ ]:
ktm = hq_utm[hq_utm["name"] == "Kathmandu"]
ktm_zone = ktm.buffer(10000)
print(type(ktm_zone))
print(ktm_zone.area.iloc[0] / 1000000)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
districts_utm.plot(ax=ax, color="lightgrey", edgecolor="white")
ktm_zone.plot(ax=ax, color="tab:blue", alpha=0.5)
ktm.plot(ax=ax, color="black", markersize=15)
ax.set_xlim(300000, 370000)
ax.set_ylim(3030000, 3100000)
ax.set_title("10 km around Kathmandu headquarters")
plt.show()

The unit is the metre, so 10,000 means 10 km. The buffer's area is about 313.65 km², very close to π × 10² ≈ 314.16 km². Step 3 explains the small difference.

`alpha=0.5` makes the buffer half transparent, so the district borders show through. `ax.set_xlim` and `ax.set_ylim` zoom in on the area around Kathmandu; the numbers are UTM eastings and northings in metres. The circle reaches well beyond Kathmandu district into its neighbours.

### Practice

**Q1 · one line:** Draw a 5 km buffer around the Pokhara headquarters and print its area in km².

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(hq_utm[hq_utm["name"] == "Pokhara"].buffer(5000).area.iloc[0] / 1000000)
```

About 78.4 km², close to π × 5² ≈ 78.5 km². Halving the radius divides the area by four.
</details>

**Q2 · two lines:** Buffer all 75 headquarters by 10 km into `zones`. Print how many buffers you get and their total area in km², rounded to a whole number.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
zones = hq_utm.buffer(10000)
print(len(zones), round(zones.area.sum() / 1000000))
```

75 buffers with a total of 23,524 km². That total counts some places more than once, where buffers overlap. Step 5 fixes this.
</details>

**Q3 · three lines:** `.buffer()` drops the names. Keep them: make a copy of `hq_utm` called `hq_zones`, replace its `geometry` column with the 10 km buffers, and show the first 3 rows.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hq_zones = hq_utm.copy()
hq_zones["geometry"] = hq_utm.buffer(10000)
hq_zones.head(3)
```

Now each buffer keeps its headquarters' name: Baglung, Baitadi, Beni. `.copy()` makes an independent copy, so `hq_utm` keeps its points.
</details>

## 2. Why metres, and not degrees or Web Mercator

Buffering in the wrong CRS gives wrong shapes, often with no error. Here are both classic mistakes, made on purpose.

- **Degrees.** In B1 you saw that 1° of latitude is about 111 km, but 1° of longitude is only about 111 × cos(latitude) km. A buffer of 0.1° is therefore not a circle on the ground but an oval, taller than it is wide. GeoPandas prints a warning, but still draws it.
- **Web Mercator (EPSG:3857).** Its unit is called the metre, but its "metres" are stretched more and more away from the equator (B4). At Kathmandu's latitude a "10 km" Web Mercator buffer is really much smaller than 10 km.

Both wrong buffers are converted to UTM 45N below, to measure them on the ground.

In [ ]:
ktm_deg = hq[hq["name"] == "Kathmandu"]                    # still in degrees
wrong_deg = ktm_deg.buffer(0.1).to_crs(32645)                   # WRONG: 0.1 degrees
wrong_merc = ktm_deg.to_crs(3857).buffer(10000).to_crs(32645)    # WRONG: Web Mercator "metres"

print("correct  ", round(ktm_zone.area.iloc[0] / 1000000, 1))
print("degrees  ", round(wrong_deg.area.iloc[0] / 1000000, 1))
print("Mercator ", round(wrong_merc.area.iloc[0] / 1000000, 1))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ktm_zone.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=2)
wrong_deg.plot(ax=ax, facecolor="none", edgecolor="tab:red", linestyle="--")
wrong_merc.plot(ax=ax, facecolor="none", edgecolor="tab:orange", linestyle="--")
ax.set_title("Correct (black), 0.1° (red), Web Mercator (orange)")
ax.set_axis_off()
plt.show()

The warning in the first cell says it plainly: results from `buffer` in a geographic CRS "are likely incorrect". The correct circle covers 313.7 km². The 0.1° buffer covers 342.7 km² and is an oval. The Web Mercator buffer covers only 244.9 km²: too small by about a fifth.

On the map, the red oval is taller than the black circle, and the orange circle is clearly smaller. Neither mistake raised an error. The rule for the whole of Part B: **buffer and measure in metres, in a projected CRS that suits Nepal, such as UTM 45N**.

### Practice

**Q1 · one line:** What radius does the Web Mercator buffer really have? Use the circle formula backwards: radius = √(area ÷ π). Print it in km. The setup cell imported `math` (A5), which gives `math.sqrt` and `math.pi`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(math.sqrt(wrong_merc.area.iloc[0] / math.pi) / 1000)
```

About 8.8 km instead of 10 km.
</details>

**Q2 · two lines:** Measure the 0.1° oval: take `wrong_deg.total_bounds` (B3) and print its width and its height in km.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
b = wrong_deg.total_bounds
print((b[2] - b[0]) / 1000, (b[3] - b[1]) / 1000)
```

About 19.7 km wide but 22.2 km tall. In the north–south direction 0.1° is about 11.1 km; east–west it is only 111 × cos(27.7°) × 0.1 ≈ 9.8 km.
</details>

**Q3 · three lines:** Print the area of the Web Mercator buffer as a percentage of the correct one, rounded to 1 decimal place. Store both areas in km² first.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
merc_km2 = wrong_merc.area.iloc[0] / 1000000
correct_km2 = ktm_zone.area.iloc[0] / 1000000
print(round(merc_km2 / correct_km2 * 100, 1))
```

78.1%: more than a fifth of the zone is missing. Web Mercator is fine for showing web maps, but never for measuring.
</details>

## 3. Rounder or flatter: `quad_segs` and cap styles

A buffer circle is stored as a polygon with many short straight sides. `quad_segs` sets how many sides make up each **quarter** of the circle. The default is 16, so a full circle has 64 sides. That is why the area in Step 1 was a little below π r².

Around a line, the **cap style** decides the shape of the two ends:

| `cap_style=` | The ends are… |
| --- | --- |
| `"round"` (default) | half circles |
| `"flat"` | cut off straight, exactly at the end points |
| `"square"` | squared, extending past the end points by the buffer distance |

The example builds the Everest trek line as in B2 and B4, and converts it to UTM 45N.

In [ ]:
for q in [1, 2, 4, 16, 32]:
    area_km2 = ktm.buffer(10000, quad_segs=q).area.iloc[0] / 1000000
    print(q, round(area_km2, 2))

In [ ]:
trek = pd.read_csv(DATA / "everest_trek.csv")
trek_line = gpd.GeoSeries([LineString(trek[["lon", "lat"]].to_numpy())], crs="EPSG:4326")
trek_utm = trek_line.to_crs(32645)
print(round(trek_utm.length.iloc[0] / 1000, 1))

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, style in zip(axes, ["round", "flat", "square"]):
    trek_utm.buffer(1000, cap_style=style).plot(ax=ax, color="tab:blue", alpha=0.4)
    trek_utm.plot(ax=ax, color="black")
    ax.set_title(f'cap_style="{style}"')
    ax.set_axis_off()
plt.show()

With `quad_segs=1` the "circle" is a square standing on its corner, with 200 km². The area grows towards π r² as the sides get more numerous: 313.65 km² at 16 and 314.03 km² at 32. The default is close enough for almost all work.

The trek line from Lukla to Everest Base Camp is about 42.4 km long, measured as straight lines between the eight stops. The three corridors differ only at the two ends: round, cut flat, or squared and extended.

### Practice

**Q1 · one line:** Print the area of the 10 km buffer around Kathmandu with `quad_segs=4`, in km².

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(ktm.buffer(10000, quad_segs=4).area.iloc[0] / 1000000)
```

About 306.1 km², a few percent below the true circle. Coarser buffers are faster to compute, but slightly too small.
</details>

**Q2 · two lines:** A flat-capped corridor is a long rectangle-like strip. Make a 1 km flat buffer of `trek_utm`, then divide its area in km² by the line's length in km. What do you get?

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
corridor = trek_utm.buffer(1000, cap_style="flat")
print(corridor.area.iloc[0] / 1000000 / (trek_utm.length.iloc[0] / 1000))
```

About 2.0 km: the full width of the corridor, 1 km on each side. Area ÷ length = width, for a flat-capped strip.
</details>

**Q3 · three lines:** Draw a 2 km flat buffer of the trek line in light blue, with the line on top in black and the trek stops as points. Make the stops with `points_from_xy` and convert them to UTM 45N.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(5, 6))
stops_utm = gpd.GeoDataFrame(trek, geometry=gpd.points_from_xy(trek["lon"], trek["lat"]), crs="EPSG:4326").to_crs(32645)
trek_utm.buffer(2000, cap_style="flat").plot(ax=ax, color="lightblue")
trek_utm.plot(ax=ax, color="black")
stops_utm.plot(ax=ax, color="black", markersize=12)
plt.show()
```

The corridor follows the line from Lukla in the south to Base Camp in the north and stops flat at both ends. All layers are in UTM 45N, so they line up.
</details>

## 4. One side only: single-sided buffers

Sometimes you need a zone on **one side** of a line only, for example the left bank of a river or the uphill side of a road. `buffer(distance, single_sided=True)` does that.

Which side? Imagine walking along the line **in the direction it was drawn**, from its first point to its last:

- a **positive** distance buffers the **left** side,
- a **negative** distance buffers the **right** side.

Some tutorials online state the opposite. Do not trust memory; test it, as below.

The trek line was drawn from Lukla (south) to Everest Base Camp (north). Walking north, your left is west. So the positive buffer should lie west of the line, with **smaller** eastings.

In [ ]:
left = trek_utm.buffer(1000, single_sided=True)     # positive: left side
right = trek_utm.buffer(-1000, single_sided=True)   # negative: right side

print("line ", round(trek_utm.centroid.x.iloc[0]))
print("left ", round(left.centroid.x.iloc[0]))
print("right", round(right.centroid.x.iloc[0]))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6))
left.plot(ax=ax, color="tab:blue", alpha=0.6)
right.plot(ax=ax, color="tab:orange", alpha=0.6)
trek_utm.plot(ax=ax, color="black")
ax.set_title("Positive = left (blue), negative = right (orange)")
ax.set_xlabel("Easting (m)")
plt.show()

The centroid of the left buffer has a smaller easting than the line, and the right buffer a larger one. Walking north from Lukla, the blue zone (positive) is on your left, the west. The test confirms the rule.

If a river line is drawn downstream, as rivers usually are, a positive single-sided buffer covers its left bank.

### Practice

**Q1 · one line:** Print the area of the left buffer `left` in km².

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(left.area.iloc[0] / 1000000)
```

About 42.8 km²: roughly 1 km times the 42.4 km length, plus a little extra where the line bends.
</details>

**Q2 · two lines:** `.reverse()` flips the direction of a line. Reverse `trek_utm`, buffer it by **+1000** single-sided, and print the easting of the result's centroid.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
backwards = trek_utm.reverse()
print(round(backwards.buffer(1000, single_sided=True).centroid.x.iloc[0]))
```

It prints 478181, the same as the right buffer before. Walking south from Base Camp, your left is east. The side depends on the line's direction, which is why you must know how a line was drawn.
</details>

**Q3 · three lines:** Make a 3 km single-sided buffer on the **east** side of the trek, as drawn from Lukla to Base Camp. Print its area in km² and draw it with the line.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
east_side = trek_utm.buffer(-3000, single_sided=True)
print(east_side.area.iloc[0] / 1000000)
east_side.plot(color="tab:orange")
plt.show()
```

Walking north, east is on the right, so the distance is negative. The area is about 121.9 km². No error would warn you if you picked the wrong sign, so check the map.
</details>

## 5. Merging overlapping buffers with `union_all()`

The 10 km zones around Kathmandu, Lalitpur and Bhaktapur overlap heavily. If you add up their areas, the overlap is counted two or three times.

`.union_all()` merges every shape in a GeoSeries into **one** shape. Overlaps are counted once, and touching or overlapping zones melt together.

The result is a single shapely shape, usually a **MultiPolygon**: one shape made of several separate polygons. It is not a GeoSeries, so it has no CRS and no `.plot()`. To draw it, wrap it: `gpd.GeoSeries([shape], crs=32645)`.

In [ ]:
zones = hq_utm.buffer(10000)
merged = zones.union_all()

print(round(zones.area.sum() / 1000000))
print(round(merged.area / 1000000))
print(merged.geom_type, len(merged.geoms))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
districts_utm.plot(ax=ax, color="lightgrey", edgecolor="white")
gpd.GeoSeries([merged], crs=32645).plot(ax=ax, color="tab:blue", alpha=0.5)
hq_utm.plot(ax=ax, color="black", markersize=4)
ax.set_title("Land within 10 km of a district headquarters")
ax.set_axis_off()
plt.show()

The 75 separate zones add up to 23,524 km², but the merged zone covers 22,650 km². About 874 km² were counted twice or more.

The merged shape is a MultiPolygon made of 67 separate pieces: where zones overlap, they have melted into one piece. Notice that `merged.area` needs no `.iloc[0]`, because `merged` is one shape, not a GeoSeries.

### Practice

**Q1 · one line:** Print the geometry type of one single buffer, `zones.iloc[0]`, to compare with the merged shape.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(zones.iloc[0].geom_type)
```

A single buffer is a `Polygon`. Merging many separate polygons gives a `MultiPolygon`.
</details>

**Q2 · two lines:** Take the 10 km zones of Kathmandu, Lalitpur and Bhaktapur only (use `.isin` on `hq_utm["name"]`). Print the sum of their areas and the area of their union, in km².

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
valley_zones = hq_utm[hq_utm["name"].isin(["Kathmandu", "Lalitpur", "Bhaktapur"])].buffer(10000)
print(valley_zones.area.sum() / 1000000, valley_zones.union_all().area / 1000000)
```

About 941 km² summed but only 573 km² merged. The three headquarters are so close that more than a third of the summed area is overlap.
</details>

**Q3 · three lines:** Repeat the comparison for 5 km zones around all 75 headquarters. Print the summed area, the merged area, and the overlap as a percentage of the summed area.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
zones_5km = hq_utm.buffer(5000)
summed = zones_5km.area.sum()
print(summed / 1000000, zones_5km.union_all().area / 1000000, (summed - zones_5km.union_all().area) / summed * 100)
```

About 5,881 km² summed and 5,832 km² merged, an overlap of under 1%. Small zones rarely meet, so the overlap shrinks quickly as the distance gets smaller.
</details>

## 6. Which landslides lie within 10 km of a headquarters?

Now combine a buffer with a predicate from B7. `slides_utm.within(merged)` asks each landslide whether it lies inside the merged 10 km zone. Both are in UTM 45N.

A buffer answers "within this distance of **any** feature". For the distance to **one** particular feature, use `.distance()`. `slides_utm.distance(point)` gives the straight-line distance from every landslide to that point, in metres.

Remember that these are straight-line distances. On a mountain road or trail, the real travel distance is much longer.

In [ ]:
near = slides_utm[slides_utm.within(merged)]
print(len(near))
print(near["district"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
districts_utm.plot(ax=ax, color="lightgrey", edgecolor="white")
gpd.GeoSeries([merged], crs=32645).plot(ax=ax, color="tab:blue", alpha=0.3)
slides_utm.plot(ax=ax, color="grey", markersize=8)
near.plot(ax=ax, color="tab:red", markersize=14)
ax.set_title("Practice landslides within 10 km of a headquarters (red)")
ax.set_axis_off()
plt.show()

In [ ]:
charikot = hq_utm.loc[hq_utm["name"] == "Charikot", "geometry"].iloc[0]
dist_km = slides_utm.distance(charikot) / 1000
print(dist_km.round(1).sort_values().head())

9 of the 40 practice landslides lie within 10 km of a headquarters: 5 in Rasuwa, 3 in Chitawan and 1 in Dolakha.

Charikot is the headquarters of Dolakha. The nearest landslide is about 6.6 km away from it in a straight line, and the next ones are more than 20 km away. The row labels (37, 30, 0, …) tell you which landslide each distance belongs to.

### Practice

**Q1 · one line:** How many landslides lie within **5 km** of a headquarters? Use the merged 5 km zone.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(slides_utm.within(hq_utm.buffer(5000).union_all()).sum())
```

Only 2. Halving the distance cut the count from 9 to 2. The answer to "how many are near?" depends on what you call near, so always state the distance.
</details>

**Q2 · two lines:** Take the point of the Dhunche headquarters (Rasuwa) and print the distance in km from it to the nearest landslide.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dhunche = hq_utm.loc[hq_utm["name"] == "Dhunche", "geometry"].iloc[0]
print((slides_utm.distance(dhunche) / 1000).min())
```

About 3.0 km. `.min()` of all 40 distances gives the nearest one.
</details>

**Q3 · three lines:** Which landslides lie within 10 km of Bharatpur (the Chitawan headquarters) only? Buffer that one point by 10 km, select the landslides within it, and print their `id` and `district`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bharatpur = hq_utm.loc[hq_utm["name"] == "Bharatpur", "geometry"].iloc[0]
near_bharatpur = slides_utm[slides_utm.within(bharatpur.buffer(10000))]
print(near_bharatpur[["id", "district"]])
```

Three landslides: LS019, LS020 and LS037, all in Chitawan. A single shapely point has its own `.buffer()`; it uses the units of the coordinates, here metres.
</details>

## 7. The nearest headquarters to every landslide: `sjoin_nearest`

Looping over 75 headquarters to find the nearest one for each landslide would be slow to write. `gpd.sjoin_nearest(left, right, distance_col="dist_m")` does it in one line.

For every row of the **left** layer, it finds the nearest feature of the **right** layer and adds that feature's columns, as a join does (B6). `distance_col="dist_m"` adds a column with the distance, in the units of the CRS.

So both layers must be in the **same projected CRS**: here UTM 45N, giving metres. The result also has a column `index_right`, the row label of the matched feature in the right layer.

The word **spatial join** means a join made by location instead of by a key column. B9 covers spatial joins in full.

In [ ]:
nearest = gpd.sjoin_nearest(slides_utm, hq_utm, distance_col="dist_m")
nearest["dist_km"] = (nearest["dist_m"] / 1000).round(1)
print(len(slides_utm), len(nearest))
nearest[["id", "district", "name", "dist_km"]].sort_values("dist_km")

Each of the 40 landslides got exactly one nearest headquarters (`name`) and its distance. The closest is LS003 in Rasuwa, 3.0 km from Dhunche. The farthest is LS027 in Gorkha, 39.9 km from Chame.

Look closely and you find a surprise: Chame is the headquarters of **Manang**, not Gorkha. LS008 in Gorkha is nearest to Besishahar (Lamjung), and LS010 in Myagdi to Jomsom (Mustang). The nearest headquarters is not always the one of the landslide's own district. Mountains, rivers and borders do not care about straight lines either, so for travel you would need a road network.

### Practice

**Q1 · one line:** Print the mean distance in km from a landslide to its nearest headquarters.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(nearest["dist_km"].mean())
```

About 20.2 km on average, for these made-up practice records.
</details>

**Q2 · two lines:** Count how many landslides have each headquarters as their nearest one, and print the result.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
per_hq = nearest["name"].value_counts()
print(per_hq)
```

Bharatpur and Charikot are each nearest to 9 landslides, Dhunche to 5, and Beni and Taplejung to 4. Eleven headquarters appear in total.
</details>

**Q3 · three lines:** `max_distance=10000` makes `sjoin_nearest` ignore anything farther than 10 km. Run it again with `max_distance=10000` and print how many landslides found a headquarters. Compare with Step 6.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
within_10km = gpd.sjoin_nearest(slides_utm, hq_utm, distance_col="dist_m", max_distance=10000)
print(len(within_10km))
print(within_10km["name"].value_counts())
```

9 landslides, the same 9 as with the merged buffer in Step 6, matched to Dhunche (5), Bharatpur (3) and Charikot (1). Two different methods giving the same answer is a strong check.
</details>

## 8. When it breaks: a shape is not a GeoSeries

`merged` from Step 5 holds the 10 km zones as one shape. This cell tries to draw it directly. Run it.

In [ ]:
merged.plot()

The last line reads: `AttributeError: 'MultiPolygon' object has no attribute 'plot'`.

An `AttributeError` (A6) means "this kind of object has no such method". `.union_all()`, `.iloc[0]` on a geometry column, and `box()` all give a single **shapely** shape. Shapely shapes can be measured and compared, but they cannot draw themselves and do not know their CRS.

The fix is to wrap the shape in a GeoSeries, stating its CRS:

`gpd.GeoSeries([merged], crs=32645).plot()`

The same wrap lets you convert a single shape with `.to_crs()` or save it to a file.

**Try it:** fix the cell above.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · How far outside are the border peaks?

In B7, the summit points of Everest, Makalu and Cho Oyu fell outside every district. Read `nepal_peaks.csv`, make the peaks into points, and convert them to UTM 45N. Use `sjoin_nearest` with `districts_utm[["district", "geometry"]]` to find each peak's nearest district and the distance in metres. Print the name, district and distance, rounded to 1 m.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
peaks_table = pd.read_csv(DATA / "nepal_peaks.csv")
peaks = gpd.GeoDataFrame(peaks_table,
                         geometry=gpd.points_from_xy(peaks_table["lon"], peaks_table["lat"]),
                         crs="EPSG:4326")
peaks_utm = peaks.to_crs(32645)

peak_district = gpd.sjoin_nearest(peaks_utm, districts_utm[["district", "geometry"]], distance_col="dist_m")
peak_district["dist_m"] = peak_district["dist_m"].round(1)
print(peak_district[["name", "district", "dist_m"]])
```

The five peaks inside a district have a distance of 0. Everest is 16.7 m outside Solukhumbu, Cho Oyu 17.9 m outside Solukhumbu, and Makalu 35.3 m outside Sankhuwasabha. These gaps are far smaller than the roughly 300 m simplification of the boundaries. Within the accuracy of the data, the summit points sit on the border line. Here the nearest district is the sensible answer.
</details>

### Review 2 · From the trek to the nearest headquarters

Make the 8 trek stops into points in UTM 45N. Use `sjoin_nearest` to find the nearest headquarters of each stop and the distance in km, rounded to 1 decimal place. Print the place, the headquarters and the distance.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
stops = gpd.GeoDataFrame(trek, geometry=gpd.points_from_xy(trek["lon"], trek["lat"]), crs="EPSG:4326")
stops_utm = stops.to_crs(32645)

stop_hq = gpd.sjoin_nearest(stops_utm, hq_utm, distance_col="dist_m")
stop_hq["dist_km"] = (stop_hq["dist_m"] / 1000).round(1)
print(stop_hq[["place", "name", "dist_km"]])
```

Salleri, the Solukhumbu headquarters, is the nearest for every stop. The straight-line distance grows from 24.1 km at Lukla to 60.6 km at Everest Base Camp. On foot, the real distance is much longer.
</details>

### Review 3 · Rings around Charikot

Count the landslides within 5, 10, 20 and 30 km of the Charikot headquarters. Use a loop over the distances, buffer the Charikot point each time, and print the distance and the count.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
charikot = hq_utm.loc[hq_utm["name"] == "Charikot", "geometry"].iloc[0]

for distance_km in [5, 10, 20, 30]:
    zone = charikot.buffer(distance_km * 1000)
    n_inside = slides_utm.within(zone).sum()
    print(distance_km, "km:", n_inside)
```

0 within 5 km, 1 within 10 km and still 1 within 20 km, then 7 within 30 km: six landslides lie in the ring between 20 and 30 km. Rings of increasing distance show how a count depends on the distance you choose.
</details>

### Review 4 · A report of remote landslides

Using `nearest` from Step 7, select the landslides more than 30 km from their nearest headquarters. Keep the columns `id`, `district`, `name` and `dist_km`, sort them from the most remote, print them, and save them to `remote_landslides.csv` in `OUT` without the index.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
remote = nearest[nearest["dist_km"] > 30]
report = remote[["id", "district", "name", "dist_km"]].sort_values("dist_km", ascending=False)
print(report)
print(len(report))

report.to_csv(OUT / "remote_landslides.csv", index=False)
print((OUT / "remote_landslides.csv").exists())
```

6 landslides are more than 30 km from any headquarters, the most remote being LS027 in Gorkha (39.9 km from Chame). The saved table has no geometry column, so it is a plain CSV file that anyone can open. These are practice records, not real events.
</details>

## Recap

- A **buffer** is the area within a distance of a feature: `gdf.buffer(10000)`. The distance is in CRS units, so convert to UTM 45N (metres) first.
- Never buffer in degrees (ovals, with a warning) or Web Mercator (about a fifth too small at Kathmandu, with no warning).
- `quad_segs` sets how round a buffer is; `cap_style="round"`, `"flat"` or `"square"` shapes the ends of a line buffer. With `single_sided=True`, a positive distance buffers the **left** side of a line, in the direction it was drawn.
- `.union_all()` merges overlapping buffers into one shape, so overlaps are counted once. Wrap a single shape in `gpd.GeoSeries([shape], crs=...)` to plot or convert it.
- `.distance()` measures to one feature; `gpd.sjoin_nearest(left, right, distance_col="dist_m")` finds the nearest feature for every row. Both measure straight lines, not travel routes.

### Words you learned

| Word | Meaning |
| --- | --- |
| buffer | The area within a given distance of a feature |
| `quad_segs` | How many straight sides make up each quarter of a buffer circle |
| cap style | The shape of the ends of a line buffer: round, flat or square |
| single-sided buffer | A buffer on one side of a line only; positive distance = left side |
| union | One shape covering everything covered by a set of shapes |
| `union_all()` | Merges all shapes of a GeoSeries into one shape |
| MultiPolygon | One shape made of several separate polygons |
| `alpha` | Transparency of a layer, from 0 (invisible) to 1 (solid) |
| straight-line distance | The shortest distance on the map, ignoring roads and terrain |
| spatial join | A join that matches rows by location instead of by a key column |
| `sjoin_nearest` | Joins each feature to the nearest feature of another layer |
| `distance_col` | The column where `sjoin_nearest` stores the distance |
| `max_distance` | The largest distance `sjoin_nearest` will search |

## Next

**B9 · Spatial joins and overlays:** join landslides to the districts they fall in, count them per district, and cut layers against each other with overlay.

<!-- pygile-nav -->
**PyGILE Course** · lesson 18 of 21 · [Course home](../README.md)  
← Previous: [B7 · Clip and select by location: inside, touching, cut to shape](B07_Clip_and_select_by_location.ipynb) · Next: [B9 · Spatial joins and overlays: landslides, districts and headquarters](B09_Spatial_joins_and_overlays.ipynb) →